In [ ]:
# ── CELL 1: MOUNT & INSTALL ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
# !pip install statsmodels -q
#!pip install regex
#import regex as re

Mounted at /content/drive


In [6]:
exec(open('/content/drive/MyDrive/HEC Thesis/extend_master_macro_pre2011.py').read())


  Loading Master_Macro.csv
  457 rows x 237 columns
  Truly new dates:        0
  Existing empty rows:    96
  Total to process:       96

  1. Implied FFR (Kuttner)
  Kuttner: 96/96 computed

  2. FRED Data
  + cpi (CPIAUCSL)
  + core_cpi (CPILFESL)
  + pce (PCEPI)
  + core_pce (PCEPILFE)
  + unemployment_rate (UNRATE)
  + nonfarm_payroll (PAYEMS)
  + gdp (GDP)
  + gdp_deflator (GDPDEF)
  + nat_unemp_rate (NROU)
  + fed_funds_rate (FEDFUNDS)
  + yield_3mo (DTB3)
  + yield_6mo (DTB6)
  + yield_2yr (DGS2)
  + yield_5yr (DGS5)
  + yield_10yr (DGS10)
  + vix (VIXCLS)
  + breakeven_10yr (T10YIE)
  + real_rate_5yr (DFII5)
  + term_spread_10_2 (T10Y2Y)
  + excess_bond_prem (BAMLC0A0CM)

  3. SPF Data
  + spf_gdp_1q (col=RGDP2)
  + spf_unemp_1q (col=UNEMP2)
  + spf_cpi_1q (col=CPI2)

  4. Computing values
  Filling 96 existing empty rows...

  5. Saving
  Saved: /content/drive/MyDrive/HEC Thesis/Data/Master_Macro.csv
  457 rows x 237 columns
  Meeting dates: 217
  Date range: 1999-02-03 to 2

In [8]:
exec(open('/content/drive/MyDrive/HEC Thesis/append_sentiment_channels_v3.py').read())


══════════════════════════════════════════════════════════════════════
  Loading Master_Macro.csv
══════════════════════════════════════════════════════════════════════
  Loaded: 457 rows × 237 columns
  Date range: 1999-02-03 → 2025-12-31
  Event types: {'meeting_date': 217, 'minutes_date': 121, 'blackout_date': 119}
  Dropped old channel columns, rebuilding from scratch.

══════════════════════════════════════════════════════════════════════
  1. Forward Guidance Shift
══════════════════════════════════════════════════════════════════════
  ✓  ffr_path_change: 456 non-NaN rows

══════════════════════════════════════════════════════════════════════
  2. Inflation Expectations
══════════════════════════════════════════════════════════════════════
  ✓  michigan_1yr_inf_exp: 457 non-NaN
  ✓  inf_exp_gap:          457 non-NaN
  ✓  inf_exp_5y5y: 425 non-NaN

══════════════════════════════════════════════════════════════════════
  3. Financial Conditions
═══════════════════════════════════

In [9]:
import os, json, pandas as pd

# ── Check 1: JSON files ───────────────────────────────────────────────────
JSON_STMTS = '/content/drive/MyDrive/HEC Thesis/Text Data/structured_json_statements'
files = os.listdir(JSON_STMTS) if os.path.exists(JSON_STMTS) else []
print(f"JSON folder exists: {os.path.exists(JSON_STMTS)}")
print(f"JSON files found: {len(files)}")
print(f"First 3: {sorted(files)[:3]}")
print(f"Last 3:  {sorted(files)[-3:]}")

# Try reading one
if files:
    fpath = os.path.join(JSON_STMTS, sorted(files)[0])
    with open(fpath) as f:
        rec = json.load(f)
    print(f"\nSample record keys: {list(rec.keys())}")
    print(f"Date: {rec['date']}  Words: {rec['word_count']}")
    print(f"Text preview: {rec['text'][:100]}")

# ── Check 2: Dissent data ─────────────────────────────────────────────────
TW_FILE = '/content/drive/MyDrive/HEC Thesis/Data/FOMC_Dissents_Data.xlsx'
tw = pd.read_excel(TW_FILE, skiprows=3)
tw.columns = tw.columns.str.strip()
tw = tw.rename(columns={'FOMC Meeting': 'date'})
tw['date'] = pd.to_datetime(tw['date']).dt.normalize()
tw = tw.rename(columns={
    'Votes Against Action':         'fomc_dissent_count',
    'Number Governors Dissenting':  'fomc_gov_dissent',
    'Number Presidents Dissenting': 'fomc_pres_dissent',
})
print(f"\nTW rows total: {len(tw)}")
print(f"TW date range: {tw['date'].min().date()} → {tw['date'].max().date()}")
print(f"TW post-1999: {(tw['date'] >= '1999-01-01').sum()} rows")

# Check overlap with master
MASTER_CSV = '/content/drive/MyDrive/HEC Thesis/Data/Master_Macro.csv'
df = pd.read_csv(MASTER_CSV, parse_dates=['date'])
mtg_dates = df[df['event_type']=='meeting_date']['date'].dt.normalize()
tw_dates  = tw['date'].dt.normalize()
print(f"\nMeeting dates in master: {len(mtg_dates)}")
print(f"TW dates: {len(tw_dates)}")
matched = mtg_dates[mtg_dates.isin(tw_dates)]
print(f"Exact matches: {len(matched)}")
print(f"\nFirst 5 master meeting dates: {sorted(mtg_dates)[:5]}")
print(f"First 5 TW dates: {sorted(tw_dates)[:5]}")

JSON folder exists: True
JSON files found: 235
First 3: ['statement_19940204.json', 'statement_19940322.json', 'statement_19940418.json']
Last 3:  ['statement_20250917.json', 'statement_20251029.json', 'statement_20251210.json']

Sample record keys: ['source', 'date', 'date_raw', 'type', 'text', 'word_count']
Date: 1994-02-04  Words: 103
Text preview: Chairman Alan Greenspan announced today that the Federal Open Market Committee decided to increase s

TW rows total: 860
TW date range: 1936-03-19 → 2026-03-18
TW post-1999: 225 rows

Meeting dates in master: 218
TW dates: 860
Exact matches: 218

First 5 master meeting dates: [Timestamp('1999-02-03 00:00:00'), Timestamp('1999-03-30 00:00:00'), Timestamp('1999-05-18 00:00:00'), Timestamp('1999-06-30 00:00:00'), Timestamp('1999-08-24 00:00:00')]
First 5 TW dates: [Timestamp('1936-03-19 00:00:00'), Timestamp('1936-05-25 00:00:00'), Timestamp('1936-11-20 00:00:00'), Timestamp('1937-01-26 00:00:00'), Timestamp('1937-03-15 00:00:00')]
